In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT
if not (PROJECT_ROOT / "SEConformer").exists():
    for parent in [CURRENT, *CURRENT.parents]:
        if (parent / "SEConformer").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("Projeto:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())


In [ ]:
from SEConformer import DEVICE, build_inbreast_csv, make_run_dirs, train_inbreast_holdout

INBREAST_CSV_SOURCE = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "INbreast.csv"
INBREAST_DICOM_DIR = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "AllDICOMs"
INBREAST_FOLDS_CSV = PROJECT_ROOT / "SEConformer" / "inbreast_birads_folds.csv"

print("Device:", DEVICE)
print("Dataset:", INBREAST_CSV_SOURCE)


In [ ]:
df = build_inbreast_csv(
    inbreast_csv_path=str(INBREAST_CSV_SOURCE),
    dicom_dir=str(INBREAST_DICOM_DIR),
    out_csv=str(INBREAST_FOLDS_CSV),
    mode="multiclass",
    n_splits=5,
    random_state=42,
)
print("Total imagens:", len(df))
df.head()


In [ ]:
run_dirs = make_run_dirs(run_prefix="seconformer_inbreast_holdout")
model, run_dirs, metrics = train_inbreast_holdout(
    csv_path=str(INBREAST_FOLDS_CSV),
    val_fraction=0.2,
    epochs=5,
    batch_size=16,
    run_dirs=run_dirs,
    img_size=224,
    balance=True,
)

print("Run:", run_dirs["out_dir"])
metrics
